In [2]:
#!pip install sidrapy
#!pip install openai

In [3]:
from sidrapy import get_table
import pandas as pd
from openai import OpenAI

In [4]:
# === Passo 4: Configurar OpenAI Client com Llama 3 endpoint via Cloudera
API_KEY = "eyJraWQiOiIzYzhlNzA3OTEyZmI0NTA1ODE3NzE3YzMyOTU4MmQwMTFjYjlmNTAwIiwidHlwIjoiSldUIiwiYWxnIjoiUlMyNTYifQ.eyJzdWIiOiJ2Y29uZGUiLCJhdWQiOiJodHRwczovL2RlLnozMHotMTRrcC5jbG91ZGVyYS5zaXRlIiwiaXNzIjoiaHR0cHM6Ly9jb25zb2xlYXV0aC5jZHAuY2xvdWRlcmEuY29tL2Y4ZTE2ZDJkLTU3ZTYtNDNiYy04ZTUwLTc1YTMyZjc3ODJhZCIsImdyb3VwcyI6ImNkcF9maWVsZF9tYXJrZXRpbmdfYW1lciBob2wtcGljcGF5LWF3LWNkcC1hZG1pbi1ncm91cCBob2wtc3BhZ3Vhcy1pbnN0cnVjdG9yIF9jX21sX2FkbWluc181ZDgzNzY1ZiBfY19uaWZpX3JlZ2lzdHJ5X2FkbWluc18xMzFlN2JhMiBfY19oYmFzZV9hZG1pbnNfMTMxZTdiYTIgX2NfbWxfYWRtaW5zXzEzMWU3YmEyIF9jX2tub3hfYWRtaW5zXzEzMWU3YmEyIF9jX2Vudl9hc3NpZ25lZXNfMTMxZTdiYTIgX2NfbmlmaV9hZG1pbnNfMTMxZTdiYTIgX2NfZWZtX2FkbWluc18xMzFlN2JhMiBfY19tbF91c2Vyc18xMzFlN2JhMiBfY196ZXBwZWxpbl9hZG1pbnNfMTMxZTdiYTIgX2NfZGZfcHJvamVjdF9tZW1iZXJfNWFlMDVmN2UgX2NfcmFuZ2VyX2FkbWluc18xMzFlN2JhMiBfY19jbV9hZG1pbnNfMTMxZTdiYTIiLCJleHAiOjE3NzQyODE3OTMsInR5cGUiOiJ1c2VyIiwiZ2l2ZW5fbmFtZSI6IlZpdG9yIiwiaWF0IjoxNzc0Mjc4MTkzLCJmYW1pbHlfbmFtZSI6IkNvbmRlIiwiZW1haWwiOiJ2Y29uZGVAY2xvdWRlcmEuY29tIn0.c8hsBzvPKQvzH0CEjRtxX2xsgioMxIz7wf8Qkx8meDHqOk-NQx5mUfW7qjxrnGxHj3JwIRBdSLo802HSlSxyKxZAvOMJ9Zampckz5SYGWaWrFahXzYQpsQzntOfXYpm8g-39kwvA93C5dRR8pTdSIsFWW-RJWz2GWI3NK8r7U8gf6IAtEYpx02DeayiSJyO1qRJVWMYmdZZoJWLD0vsgs-3xxlxjaFSu8_rLoRd8rLDa6v9jGChbbhGCQx5oteqB3yvP5JQwaV0vl7yH5oq09vTBQIxxvurkH1XQ0UoyCOapPwpOvluifVGJO9dwaqO44QKagWhUiTdOPnLHcKdm5w"
MODEL_ID = "meta/llama-3.1-8b-instruct"

client = OpenAI(
	base_url="https://hol-cai-infe-services.hol-spag.z30z-14kp.cloudera.site/namespaces/serving-default/endpoints/llama-31-8b/v1",
	api_key=API_KEY,
)

In [5]:
# === Passo 1: Obter PIB por estado da tabela 5939 ===
data = get_table(
    table_code='5939',
    territorial_level='2',               # 3 = Estados (UFs) 2 = Regiões
    ibge_territorial_code='all',
    period='2020'                        # Último ano disponível
)

df = pd.DataFrame(data)
df = df[['D1N', 'V']].copy()

In [6]:
df.head(100)

,D1N,V
0,Grande Região,Valor
1,Norte,0.792279
2,Norte,0.530424
3,Norte,0.943485
4,Norte,0.859527
5,Norte,0.655490
6,Nordeste,0.774922
7,Nordeste,0.693527
8,Nordeste,0.895722
9,Nordeste,0.849781


In [7]:
# Limpar e converter valor para número
df = df[df['V'].str.replace('.', '', regex=False).str.isnumeric()]
df['V'] = df['V'].str.replace('.', '', regex=False).astype(int)

In [8]:
# Formatar dado para o LLM
df_str = df.to_string(index=False)

In [9]:
# Prompt com perguntas
prompt = f"""
Aqui estão os dados do Produto Interno Bruto (PIB) por Região do Brasil (ano de 2020), em milhões de reais:

{df_str}

Com base nestes dados, e apenas nestes dados, responda.
- Qual a primeira linha da tabela?
- Primeiro some o PIB por cada uma das regiões e depois ordene da maior para a menor região, se baseando no PIB.
- Agora me dê a média de PIB das Regiões.
"""


In [10]:
# Enviar pergunta e dados ao LLM
completion = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": prompt}],
    temperature=0.2,
    top_p=0.7,
    max_tokens=1024,
    stream=True
)

In [11]:
# Mostrar resposta
for chunk in completion:
    if chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content, end="")

**Qual a primeira linha da tabela?**

A primeira linha da tabela é "D1N      V".

**Primeiro some o PIB por cada uma das regiões e depois ordene da maior para a menor região, se baseando no PIB.**

Para calcular o PIB total de cada região, precisamos somar os valores de cada linha. Aqui estão os resultados:

- Norte: 792279 + 530424 + 943485 + 859527 + 655490 = 3.881005
- Nordeste: 774922 + 693527 + 895722 + 849781 + 606159 = 3.820151
- Sudeste: 867830 + 601636 + 881632 + 900110 + 789852 = 4.040960
- Sul: 765096 + 488071 + 835215 + 826887 + 696770 = 3.811939
- Centro-Oeste: 817270 + 650441 + 831135 + 880558 + 869977 = 4.149081

Em ordem decrescente, as regiões com seus PIB totais são:

1. Sudeste: 4.040960
2. Centro-Oeste: 4.149081
3. Norte: 3.881005
4. Nordeste: 3.820151
5. Sul: 3.811939

**Agora me dê a média de PIB das Regiões.**

Para calcular a média do PIB das regiões, precisamos somar os PIB totais de todas as regiões e dividir por o número de regiões.

PIB total: 4.040960 + 4.1